# Stage 4/5:RC Design + 強柱弱梁——第一次真正接上 `Stage 2/3` 的需求

**這裡的結果是「`Stage 4` 強度設計結果」,不是「最終施工配筋」**——
完整的耐震設計還需要正負彎矩要求、梁端連續筋、箍筋/圍束、`capacity
-design shear`、梁柱接頭、錨定搭接、間距等一系列 `Stage 5` 檢核都
通過之後,才能正式 `Design Freeze`。這一課只完成 `PM interaction`+
最小鋼筋比+`SCWB` 這幾項,不是完整清單。

`Case-04.7`(`Stage 2/3`)完成了四工具(`OpenSeesPy`/`frame2d`/`PyNite`/
`PyFEM`)驗證過的 `Canonical Elastic Model`,算出了真正可信的
`governing demand`。這一課第一次把這組需求接進正式的 `RC` 設計流程
——不是延續 `Case-06` 系列從頭到尾都沒變過的 $\rho=0.02$ 假設值,
是真正從需求反推出配筋。

**這不是延續 `case03_7`**——`case03_7` 用的是自己獨立的簡化剪力構架
需求(`Mu≈22.6`),不是 `Stage 2/3` 四工具驗證過的版本,兩者是不同
資料來源,不該混用。

## 第 1 課:柱配筋設計——用固定斷面搜尋最小滿足需求的配筋比

斷面(`40×40cm`)是 `Stage 2/3` 已確立的初步試設,這裡不重新搜尋
斷面尺寸,只搜尋配筋量。

In [1]:

import os
if not os.path.exists('rc_design.py'):
    !wget -q -O rc_design.py https://raw.githubusercontent.com/zhixiu0223/taiwan-seismic-code-calc/main/rc_design.py

from rc_design import design_column_PM, design_rebar

def Mn_at_Pu(curve, Pu_target):
    sorted_curve = sorted(curve, key=lambda p: p['Pn'])
    for i in range(len(sorted_curve)-1):
        if sorted_curve[i]['Pn'] <= Pu_target <= sorted_curve[i+1]['Pn']:
            t = (Pu_target-sorted_curve[i]['Pn'])/(sorted_curve[i+1]['Pn']-sorted_curve[i]['Pn'])
            return sorted_curve[i]['Mn'] + t*(sorted_curve[i+1]['Mn']-sorted_curve[i]['Mn'])
    return None

def design_column_rebar(b_cm, h_cm, Pu_kN, Mu_kNm, fc=280.0, fy=4200.0, Es=2.0e6,
                          cover_to_center=4.0, rho_candidates=None):
    """柱配筋設計迴圈: 給定固定斷面, 從真實需求反推出剛好滿足需求
    的最小配筋比——不是延續假設值。候選清單從ACI 318柱最小配筋比
    (1%)開始, 因為規範不允許低於這個值, 不是demand本身的限制。"""
    if rho_candidates is None:
        rho_candidates = [0.010, 0.012, 0.014, 0.016, 0.018, 0.020, 0.025, 0.030, 0.040]
    for rho in rho_candidates:
        As_bar = rho*b_cm*h_cm/8
        As_layers = [3*As_bar, 2*As_bar, 3*As_bar]
        r = design_column_PM(b_cm=b_cm, h_cm=h_cm, As_per_layer_cm2=As_layers,
                               cover_to_center_cm=cover_to_center, n_layers=3,
                               fc=fc, fy=fy, Es=Es, Pu_kN=Pu_kN, Mu_kNm=Mu_kNm)
        if r['within_envelope']:
            return dict(rho=rho, As_total=8*As_bar, r=r,
                        Mn=Mn_at_Pu(r['curve'], Pu_kN))
    return None

# governing demand來自Case-04.7(Stage 2/3, 四工具驗證過)
print("=== 1F柱(governing: Pu=308.85kN, Mu=32.13kN-m) ===")
result_1F = design_column_rebar(40.0, 40.0, 308.85, 32.13)
print(f"最小滿足需求的rho = {result_1F['rho']:.3%}")
print(f"As_total = {result_1F['As_total']:.2f}cm^2, Mn = {result_1F['Mn']:.2f}kN-m")
print(f"utilization = {result_1F['r']['utilization']:.4f}")

print()
print("=== 2F柱(governing: Pu=153.19kN, Mu=11.06kN-m) ===")
result_2F = design_column_rebar(40.0, 40.0, 153.19, 11.06)
print(f"最小滿足需求的rho = {result_2F['rho']:.3%}")
print(f"As_total = {result_2F['As_total']:.2f}cm^2, Mn = {result_2F['Mn']:.2f}kN-m")
print(f"utilization = {result_2F['r']['utilization']:.4f}")

# 確認demand本身允許的下限(避免候選清單邊界誤導判斷)
print()
print("=== 驗證: 最小配筋比是不是被規範鎖住, 不是被demand本身要求 ===")
for rho_test in [0.004, 0.006, 0.008]:
    As_bar_t = rho_test*40.0*40.0/8
    As_layers_t = [3*As_bar_t, 2*As_bar_t, 3*As_bar_t]
    r_t = design_column_PM(b_cm=40.0, h_cm=40.0, As_per_layer_cm2=As_layers_t,
                             cover_to_center_cm=4.0, n_layers=3,
                             fc=280.0, fy=4200.0, Es=2.0e6, Pu_kN=308.85, Mu_kNm=32.13)
    print(f"  rho={rho_test:.3%}: within_envelope={r_t['within_envelope']}, utilization={r_t['utilization']:.4f}")
print("即使rho=0.4%都還滿足需求——這代表真正governing的是ACI 318最小")
print("配筋比規範(1%), 不是強度需求本身; 這根柱斷面(40x40cm)相對真實")
print("需求來說偏大, 容量遠超實際需要")


=== 1F柱(governing: Pu=308.85kN, Mu=32.13kN-m) ===
最小滿足需求的rho = 1.000%
As_total = 16.00cm^2, Mn = 159.97kN-m
utilization = 0.2097

=== 2F柱(governing: Pu=153.19kN, Mu=11.06kN-m) ===
最小滿足需求的rho = 1.000%
As_total = 16.00cm^2, Mn = 136.83kN-m
utilization = 0.0818

=== 驗證: 最小配筋比是不是被規範鎖住, 不是被demand本身要求 ===
  rho=0.400%: within_envelope=True, utilization=0.2307
  rho=0.600%: within_envelope=True, utilization=0.2279
  rho=0.800%: within_envelope=True, utilization=0.2127
即使rho=0.4%都還滿足需求——這代表真正governing的是ACI 318最小
配筋比規範(1%), 不是強度需求本身; 這根柱斷面(40x40cm)相對真實
需求來說偏大, 容量遠超實際需要


## 第 2 課:梁配筋設計——同樣用 `Stage 2/3` 的真實需求

**誠實記錄一個重要釐清**:這裡算出的梁需求(`Mu≈24.19`/`16.78`)
遠低於 `VL-14` 當時反推的值(`Mu=428.44`)——**這不是矛盾,是兩者
代表完全不同的分析層級**:這裡用的是規範等效靜力法(設計地震力)
下的線彈性需求,是標準 `RC` 設計規範該用的層級;`VL-14` 用的是
`pushover` 側推分析推到 `4%` 層間位移(遠超設計地震力)時的極限
層級需求。標準 `RC` 設計只用規範地震力做配筋,不會用側推分析的
極限內力去決定該配多少鋼筋——`VL-14` 當時「從 `pushover` 結果反推
配筋」這個做法,事後看來混用了設計層級跟驗證層級的需求,兩者都有
各自正確的用途,不是誰對誰錯。

In [2]:

print("=== 1F樑(governing: Mu=24.19kN-m) ===")
r_beam_1F = design_rebar(24.19, 30.0, 50.0, cover=4.0)
print(f"rho_req(強度需求) = {r_beam_1F['rho_req']:.5f}")
print(f"rho_min(最小鋼筋比, ACI 318標準公式max(14/fy, 0.8*sqrt(fc')/fy)) = {r_beam_1F['rho_min']:.5f}")
print(f"governing: {'最小鋼筋比' if r_beam_1F['rho_min'] > r_beam_1F['rho_req'] else '強度需求'}")
print(f"As_provided={r_beam_1F['As_provided']:.2f}cm^2, phiMn={r_beam_1F['phiMn_provided']:.2f}")
print(f"bar_size={r_beam_1F['bar_size']}, n_bars={r_beam_1F['n_bars']}")

print()
print("=== 屋頂樑(governing: Mu=16.78kN-m) ===")
r_beam_roof = design_rebar(16.78, 30.0, 50.0, cover=4.0)
print(f"rho_req(強度需求) = {r_beam_roof['rho_req']:.5f}")
print(f"rho_min(最小鋼筋比) = {r_beam_roof['rho_min']:.5f}")
print(f"governing: {'最小鋼筋比' if r_beam_roof['rho_min'] > r_beam_roof['rho_req'] else '強度需求'}")
print(f"As_provided={r_beam_roof['As_provided']:.2f}cm^2, phiMn={r_beam_roof['phiMn_provided']:.2f}")
print(f"bar_size={r_beam_roof['bar_size']}, n_bars={r_beam_roof['n_bars']}")

print()
print("[誠實記錄] 兩根梁都被最小鋼筋比govern, 不是強度需求算出來的")
print("結果——這不是設計函式漏掉最小鋼筋比檢查, 是design_rebar()")
print("本身已經內建這個規範要求(rho_min), 只是上一版沒有明確呈現")
print("這層governing關係, 容易讓人誤以為只是單純強度算出的配筋")


=== 1F樑(governing: Mu=24.19kN-m) ===
rho_req(強度需求) = 0.00115
rho_min(最小鋼筋比, ACI 318標準公式max(14/fy, 0.8*sqrt(fc')/fy)) = 0.00333
governing: 最小鋼筋比
As_provided=5.73cm^2, phiMn=89.45
bar_size=#6(D19), n_bars=2

=== 屋頂樑(governing: Mu=16.78kN-m) ===
rho_req(強度需求) = 0.00079
rho_min(最小鋼筋比) = 0.00333
governing: 最小鋼筋比
As_provided=5.73cm^2, phiMn=89.45
bar_size=#6(D19), n_bars=2

[誠實記錄] 兩根梁都被最小鋼筋比govern, 不是強度需求算出來的
結果——這不是設計函式漏掉最小鋼筋比檢查, 是design_rebar()
本身已經內建這個規範要求(rho_min), 只是上一版沒有明確呈現
這層governing關係, 容易讓人誤以為只是單純強度算出的配筋


## 第 3 課:強柱弱梁檢核(`Stage 5`)——`ACI 318` 對特殊抗彎構架的規範要求

$\sum M_{nc} \geq 1.2\sum M_{nb}$,每個接頭都要檢核。用剛設計出的
柱(`1F`,`rho=1%`)跟梁(`1F`,`2-D19`)在 `1F` 接頭做示範。

In [3]:

Mn_col_1F = result_1F['Mn']
Mn_col_2F = result_2F['Mn']
Mn_beam_1F = r_beam_1F['phiMn_provided']/0.9  # 反推標稱值(phi=0.9拉力控制)

# 修正記錄: 原本這裡用2*Mn_col_1F(1F柱容量算兩次)當sum_Mnc, 註解寫
# "保守示範"——但這個方向判斷是錯的: 1F柱(軸力較大)的Mn本來就比
# 2F柱大, 用2倍1F柱容量反而"高估"了真正的sum(Mnc), 讓SCWB比值
# 比真實情況更容易通過, 不是保守。正確做法是用接頭上下真正匯入的
# 兩根柱(1F柱+2F柱, 軸力不同、容量不同), 不是同一根柱算兩次
sum_Mnc = Mn_col_1F + Mn_col_2F
sum_Mnb = Mn_beam_1F

ratio = sum_Mnc/sum_Mnb
print(f"柱標稱容量Mn = {Mn_col_1F:.2f}kN-m")
print(f"梁標稱容量Mn = {Mn_beam_1F:.2f}kN-m")
print(f"sum(Mnc) = {sum_Mnc:.2f}, 1.2*sum(Mnb) = {1.2*sum_Mnb:.2f}")
print(f"比值 = {ratio:.2f}(規範要求 >= 1.2)")

assert ratio >= 1.2, "強柱弱梁檢核應該通過"
print(f"\n[PASS] 強柱弱梁檢核通過, 餘裕{ratio:.2f}倍")
print()
print("對照Case-06.6(用假設rho=0.02柱+VL-14反推的超大梁配筋)算出")
print("\"柱先降伏\"(強梁弱柱)這個結果——這次用一致的規範需求層級")
print("設計出的柱梁配筋, 強柱弱梁大幅通過, 方向完全相反。這印證了")
print("Case-06.6那次結果建立在不一致demand層級混用之上, 不是規範")
print("設計流程該有的健康結果")


柱標稱容量Mn = 159.97kN-m
梁標稱容量Mn = 99.39kN-m
sum(Mnc) = 296.80, 1.2*sum(Mnb) = 119.27
比值 = 2.99(規範要求 >= 1.2)

[PASS] 強柱弱梁檢核通過, 餘裕2.99倍

對照Case-06.6(用假設rho=0.02柱+VL-14反推的超大梁配筋)算出
"柱先降伏"(強梁弱柱)這個結果——這次用一致的規範需求層級
設計出的柱梁配筋, 強柱弱梁大幅通過, 方向完全相反。這印證了
Case-06.6那次結果建立在不一致demand層級混用之上, 不是規範
設計流程該有的健康結果


## 第 4 課:梁的正負彎矩配筋要求(`ACI 318` 特殊抗彎構架規定)

耐震梁端承受反覆載重(正向地震跟反向地震方向相反),兩側(上緣/下緣)
都需要足夠的連續鋼筋,不能只做單向配筋設計。這裡從 `Case-04.7`
(`Stage 2/3`)重新抓出**帶正負號**的梁端彎矩(不是之前用的 `|M|`
包絡值),分別設計上緣(負彎矩)跟下緣(正彎矩)配筋。

**誠實記錄一個模型局限性**:`D+L`(純重力)情況下,這個模型算出的
梁端彎矩幾乎是 `0`——這是因為 `Stage 2/3` 的重力載重集中施加在
柱頂節點,沒有以分佈載重的形式直接作用在梁上,梁本身沒有承受真實
存在的「跨中正彎矩、支承處負彎矩」這種典型連續梁行為。這代表這
一課算出的正負彎矩需求,幾乎完全由側推力決定,剛好大小相等——
這是模型簡化造成的巧合,不是耐震設計的通例,拿掉「梁上真的有分佈
重力載重」這個簡化之後,正負彎矩不見得會再相等。

In [4]:

try:
    import openseespy.opensees as ops
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops

# 完全沿用Case-04.7(Stage 2/3)的幾何/材料/斷面參數, 這份notebook是
# 獨立檔案, 沒有繼承那邊的變數定義, 這裡重新宣告一次保持一致
L_bay = 6.0; h1 = 3.5; h2 = 3.5
h_col = 0.40; b_beam, h_beam = 0.30, 0.50
E_rc = 2.463e7
Ig_col = h_col**4/12; Ig_beam = b_beam*h_beam**3/12
Ic = 0.7*Ig_col; Ib = 0.35*Ig_beam
Ac = h_col**2; Ab = b_beam*h_beam
P_2F = 147.60; P_1F_extra = 147.60
F1_Y, F2_Y = 9.938, 15.900

def build():
    ops.wipe(); ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn', 1, 1, 3, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 5, 3, 4, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 6, 5, 6, Ab, E_rc, Ib, 1)

def run_case(loads):
    build()
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    for (n, fx, fy, mz) in loads:
        ops.load(n, fx, fy, mz)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    ops.analyze(1)
    return ops.eleForce(5), ops.eleForce(6)

f5_g, f6_g = run_case([(3,0,-P_1F_extra,0), (4,0,-P_1F_extra,0), (5,0,-P_2F,0), (6,0,-P_2F,0)])
f5_l, f6_l = run_case([(3,F1_Y,0,0), (5,F2_Y,0,0)])

Mi_DLE = f5_g[2]+f5_l[2]; Mj_DLE = f5_g[5]+f5_l[5]
Mi_DLnE = f5_g[2]-f5_l[2]; Mj_DLnE = f5_g[5]-f5_l[5]
print(f"1F樑 D+L+E: Mi={Mi_DLE:.3f}, Mj={Mj_DLE:.3f}")
print(f"1F樑 D+L-E: Mi={Mi_DLnE:.3f}, Mj={Mj_DLnE:.3f}")

# 上緣(負彎矩)取D+L+E那一組(Mi更負), 下緣(正彎矩)取D+L-E那一組
Mu_neg = abs(min(Mi_DLE, Mj_DLE))
Mu_pos = abs(max(Mi_DLnE, Mj_DLnE))

r_neg = design_rebar(Mu_neg, 30.0, 50.0, cover=4.0)
r_pos = design_rebar(Mu_pos, 30.0, 50.0, cover=4.0)
print(f"\n上緣(負彎矩, Mu={Mu_neg:.3f})配筋: As={r_neg['As_provided']:.2f}cm^2, {r_neg['bar_size']}x{r_neg['n_bars']}")
print(f"下緣(正彎矩, Mu={Mu_pos:.3f})配筋: As={r_pos['As_provided']:.2f}cm^2, {r_pos['bar_size']}x{r_pos['n_bars']}")

Mn_neg_1F = r_neg['phiMn_provided']/0.9
Mn_pos_1F = r_pos['phiMn_provided']/0.9
ratio_a = Mn_pos_1F/Mn_neg_1F
print(f"\n[ACI 318檢核a] Mn+/Mn- = {ratio_a:.3f}(規範要求 >= 0.5) {'PASS' if ratio_a >= 0.5 else 'FAIL'}")

# 檢核b: 任意位置容量 >= 0.25*max(接頭面容量)——這裡假設全長同一組
# 配筋(彈性均質梁模型沒有沿長度變化配筋這個細節), 自動滿足
max_Mn_face = max(Mn_neg_1F, Mn_pos_1F)
print(f"[ACI 318檢核b] 假設全長同一組配筋, 任意位置容量={min(Mn_neg_1F,Mn_pos_1F):.2f} "
      f">= 0.25*{max_Mn_face:.2f}={0.25*max_Mn_face:.2f}? "
      f"{'PASS' if min(Mn_neg_1F,Mn_pos_1F) >= 0.25*max_Mn_face else 'FAIL'}")
print("(這個檢核在\"全長不變配筋\"這個簡化下必然通過, 真正有意義的")
print(" 情境是梁中央有意減少鋼筋量節省成本時, 這個模型還沒有處理這件事)")


1F樑 D+L+E: Mi=-24.189, Mj=-24.168
1F樑 D+L-E: Mi=24.189, Mj=24.168

上緣(負彎矩, Mu=24.189)配筋: As=5.73cm^2, #6(D19)x2
下緣(正彎矩, Mu=24.189)配筋: As=5.73cm^2, #6(D19)x2

[ACI 318檢核a] Mn+/Mn- = 1.000(規範要求 >= 0.5) PASS
[ACI 318檢核b] 假設全長同一組配筋, 任意位置容量=99.39 >= 0.25*99.39=24.85? PASS
(這個檢核在"全長不變配筋"這個簡化下必然通過, 真正有意義的
 情境是梁中央有意減少鋼筋量節省成本時, 這個模型還沒有處理這件事)


## 第 5 課:`Capacity-Design Shear`——塑鉸機制形成後的剪力設計值

耐震剪力設計不能只用彈性分析算出的 $V_u$ 去設計箍筋——如果梁端
真的在大震下形成塑鉸,鋼筋會強化超過標稱降伏強度(`ACI 318` 用
`1.25fy` 代表這個機率超強),此時剪力需求會遠高於線彈性分析預測
的值。用剛設計出的梁端配筋,反推機率彎矩容量 $M_{pr}$,再用靜力
平衡算出塑鉸機制形成後的剪力設計值。

In [5]:

def compute_Mpr(As_cm2, b_cm, d_cm, fc=280.0, fy=4200.0):
    """機率彎矩容量: 用1.25fy(反映真實鋼筋強化超強), phi=1.0
    (評估真實可能發生的容量, 不是保守設計值)。fy單位kgf/cm^2
    (跟這個repo design_column_PM()一致的單位慣例), 換算kN-m用
    正確係數9.80665e-5(1 kgf-cm = 9.80665e-5 kN-m)。"""
    fy_pr = 1.25*fy
    a = As_cm2*fy_pr/(0.85*fc*b_cm)
    Mpr_kgfcm = As_cm2*fy_pr*(d_cm - a/2)
    return Mpr_kgfcm * 9.80665e-5

Ln = L_bay - h_col  # 淨跨, 扣除柱寬度(ACI 318標準做法)

Mpr_top_1F = compute_Mpr(r_neg['As_provided'], 30.0, r_neg['d'])
Mpr_bot_1F = compute_Mpr(r_pos['As_provided'], 30.0, r_pos['d'])

# 兩端同時形成塑鉸(方向相反, 這是最不利的機制情境)
Ve_hinge_1F = (Mpr_top_1F + Mpr_bot_1F)/Ln

print(f"淨跨Ln = {Ln}m")
print(f"1F樑 Mpr(上緣) = {Mpr_top_1F:.2f}kN-m, Mpr(下緣) = {Mpr_bot_1F:.2f}kN-m")
print(f"純塑鉸機制剪力 Ve_hinge = (Mpr_top+Mpr_bot)/Ln = {Ve_hinge_1F:.2f}kN")

Vu_elastic_1F = 8.06  # Stage 2/3線彈性分析算出的Vu(1F樑)
ratio_shear = Ve_hinge_1F/Vu_elastic_1F
print(f"\n對照Stage 2/3彈性需求剪力 Vu = {Vu_elastic_1F}kN")
print(f"Capacity-design shear是彈性需求的 {ratio_shear:.2f} 倍")

assert ratio_shear > 2.0, "capacity-design shear應該遠大於彈性需求(耐震設計的典型現象)"
print(f"\n[PASS] Capacity-design shear({Ve_hinge_1F:.2f}kN)遠大於彈性需求")
print(f"({Vu_elastic_1F}kN), 差了{ratio_shear:.1f}倍——這正是耐震設計")
print("\"能力保護設計\"的核心精神: 箍筋設計必須用這個較大的值, 不能")
print("只用彈性分析的Vu, 否則梁端塑鉸形成後可能發生比彎曲降伏更")
print("危險的脆性剪力破壞")


淨跨Ln = 5.6m
1F樑 Mpr(上緣) = 123.00kN-m, Mpr(下緣) = 123.00kN-m
純塑鉸機制剪力 Ve_hinge = (Mpr_top+Mpr_bot)/Ln = 43.93kN

對照Stage 2/3彈性需求剪力 Vu = 8.06kN
Capacity-design shear是彈性需求的 5.45 倍

[PASS] Capacity-design shear(43.93kN)遠大於彈性需求
(8.06kN), 差了5.5倍——這正是耐震設計
"能力保護設計"的核心精神: 箍筋設計必須用這個較大的值, 不能
只用彈性分析的Vu, 否則梁端塑鉸形成後可能發生比彎曲降伏更
危險的脆性剪力破壞


## 第 6 課:柱端圍束設計(`ACI 318-19 18.7.5`,特殊抗彎構架柱橫向鋼筋)

柱端塑鉸區需要密集的橫向鋼筋(箍筋),提供核心混凝土圍束、防止主筋
挫屈——這是特殊抗彎構架跟一般構架柱最關鍵的差異之一。三項要求:
圍束區長度、箍筋間距上限、橫向鋼筋量下限。這裡用 `1F` 柱(`40×40cm`,
剛設計出的 $\rho=1\%$ 配筋)示範。

**這裡用的是簡化的正方形對稱矩形箍情境**,沒有處理交叉箍筋
(`crosstie`)這種更複雜的排列方式,是這個計算的簡化範圍,不是完整
的排筋細部設計。

In [6]:

def design_column_confinement(h_col_cm, Ln_cm, db_long_cm, fc=280.0, fyt=4200.0,
                                 cover_cm=4.0):
    """柱端塑鉸區橫向鋼筋設計(ACI 318-19 18.7.5)。簡化情境:
    正方形對稱斷面, 單一矩形箍, 沒有處理交叉箍筋排列細節。"""

    # (1) 圍束區長度lo = max(h, Ln/6, 45cm[約18in])
    lo = max(h_col_cm, Ln_cm/6, 45.0)

    # (2) 箍筋間距上限: min(h/4, 6*db_long, so)
    #     so = 10+(35-hx)/3, 限制在10~15cm(公制近似值), hx簡化假設
    #     約等於核心淨寬(沒有中間交叉筋這個保守假設)
    bc = h_col_cm - 2*cover_cm
    hx = bc
    so = min(max(10.0 + (35.0-hx)/3.0, 10.0), 15.0)
    s_max = min(h_col_cm/4, 6*db_long_cm, so)

    # (3) 橫向鋼筋量Ash/s需求, 取兩式較大值
    Ag = h_col_cm**2
    Ach = bc**2
    Ash_over_s_a = 0.3*bc*(fc/fyt)*(Ag/Ach - 1)
    Ash_over_s_b = 0.09*bc*fc/fyt
    Ash_over_s_req = max(Ash_over_s_a, Ash_over_s_b)

    return dict(lo=lo, s_max=s_max, bc=bc, Ag=Ag, Ach=Ach,
                Ash_over_s_a=Ash_over_s_a, Ash_over_s_b=Ash_over_s_b,
                Ash_over_s_req=Ash_over_s_req)

h_col_conf = 40.0
Ln_1F_conf = (h1 - h_beam)*100  # 柱淨高(cm): 樓層高減梁深
db_long = 1.6  # cm, 估計主筋直徑(對應rho=1%配筋量級)

r_conf = design_column_confinement(h_col_conf, Ln_1F_conf, db_long)
print(f"圍束區長度 lo = {r_conf['lo']:.1f}cm")
print(f"箍筋最大間距 s_max = {r_conf['s_max']:.1f}cm(由主筋直徑限制6*db決定)")
print(f"核心尺寸 bc = {r_conf['bc']:.1f}cm, Ag={r_conf['Ag']:.0f}cm^2, Ach={r_conf['Ach']:.0f}cm^2")
print(f"Ash/s需求(取a,b兩式較大值) = {r_conf['Ash_over_s_req']:.4f}cm^2/cm")
print(f"  (a式,核心比控制: {r_conf['Ash_over_s_a']:.4f}, b式,基本下限: {r_conf['Ash_over_s_b']:.4f})")

As_tie_D10 = 0.7133  # D10單肢面積cm^2(常見箍筋規格)
Ash_provided_2leg = 2*As_tie_D10
s_from_As = Ash_provided_2leg/r_conf['Ash_over_s_req']
s_final = min(s_from_As, r_conf['s_max'])

print(f"\n若用D10雙肢箍(Ash={Ash_provided_2leg:.2f}cm^2/道):")
print(f"  滿足鋼筋量需求的間距 = {s_from_As:.1f}cm")
print(f"  排列限制的最大間距 = {r_conf['s_max']:.1f}cm")
print(f"  實際採用間距(取較小值) = {s_final:.1f}cm")

assert s_final < r_conf['s_max'], "這個案例鋼筋量需求應該比排列限制更嚴格"
print(f"\n[PASS] 這個案例中, 橫向鋼筋量需求(核心比Ag/Ach={r_conf['Ag']/r_conf['Ach']:.2f})")
print("比排列間距限制更嚴格——這代表這根柱斷面(40x40cm)的保護層")
print("混凝土占總面積比例相當顯著, 需要較密的箍筋補償保護層脫落後")
print("核心承載力的損失, 不是隨便挑一個間距就夠")


圍束區長度 lo = 50.0cm
箍筋最大間距 s_max = 9.6cm(由主筋直徑限制6*db決定)
核心尺寸 bc = 32.0cm, Ag=1600cm^2, Ach=1024cm^2
Ash/s需求(取a,b兩式較大值) = 0.3600cm^2/cm
  (a式,核心比控制: 0.3600, b式,基本下限: 0.1920)

若用D10雙肢箍(Ash=1.43cm^2/道):
  滿足鋼筋量需求的間距 = 4.0cm
  排列限制的最大間距 = 9.6cm
  實際採用間距(取較小值) = 4.0cm

[PASS] 這個案例中, 橫向鋼筋量需求(核心比Ag/Ach=1.56)
比排列間距限制更嚴格——這代表這根柱斷面(40x40cm)的保護層
混凝土占總面積比例相當顯著, 需要較密的箍筋補償保護層脫落後
核心承載力的損失, 不是隨便挑一個間距就夠


## 第 7 課:梁柱接頭剪力設計(`ACI 318-19 Chapter 18.8`)

這是這一系列規範檢核裡最後一個結構性項目。用梁端達到 $M_{pr}$ 時的
鋼筋拉力,反推接頭核心區的水平剪力需求,再跟接頭混凝土本身的剪力
容量比較。**每一步都用 `Python` 逐步計算、印出檢查**,不是心算——
這輪 `Mpr` 計算已經踩過一次單位換算的坑,這裡刻意拆得更細。

In [7]:

# --- 1. 接頭水平剪力需求 Vj = T1 - Vcol ---
As_top = r_neg['As_provided']  # cm^2, 1F樑上緣配筋(第4課設計結果)
fy_pr = 1.25*4200.0
T1_kgf = As_top*fy_pr
T1_kN = T1_kgf*9.80665e-3
print(f"T1 = As_top*1.25*fy = {As_top}*{fy_pr} = {T1_kgf:.1f}kgf = {T1_kN:.2f}kN")

# Vcol: 子構架平衡估計, 假設反曲點在上下柱樓層中點(標準側推分析假設)
Vcol_kN = Mpr_top_1F/((h1+h2)/2)
print(f"Vcol(子構架平衡估計) = Mpr/((h1+h2)/2) = {Mpr_top_1F:.2f}/{(h1+h2)/2} = {Vcol_kN:.2f}kN")

Vj_kN = T1_kN - Vcol_kN
print(f"\nVj(接頭水平剪力需求) = T1 - Vcol = {T1_kN:.2f} - {Vcol_kN:.2f} = {Vj_kN:.2f}kN")

# --- 2. 接頭剪力容量 Vn = gamma*sqrt(fc')*Aj(ACI 318原始公式用psi制,
#     逐步轉換單位避免出錯) ---
# 注意: h_col/b_beam是這份notebook全域的"米"制變數(0.40/0.30),
# 這裡的公式要用"公分"制, 必須明確換算, 不能直接複用全域變數
# (之前這裡直接複用h_col/b_beam, 算出bj=0.4cm這種離譜的公分數字,
# 被下面的assert正確攔下, 這裡修正成明確的公分制變數)
h_col_cm2, b_beam_cm2 = h_col*100, b_beam*100
m_edge = (h_col_cm2 - b_beam_cm2)/2
bj = min(b_beam_cm2+h_col_cm2, b_beam_cm2+2*m_edge)
Aj_cm2 = bj*h_col_cm2
print(f"\n有效接頭寬度 bj = min({b_beam_cm2+h_col_cm2}, {b_beam_cm2+2*m_edge}) = {bj}cm")
print(f"接頭面積 Aj = {Aj_cm2}cm^2")

fc_psi = 280.0*14.223
Aj_in2 = Aj_cm2/6.4516
print(f"fc' = 280kgf/cm^2 = {fc_psi:.1f}psi, Aj = {Aj_in2:.2f}in^2")

gamma = 10  # 外側接頭(單邊梁匯入, 無側向梁圍束), ACI 318最不利分類
Vn_lb = gamma*(fc_psi**0.5)*Aj_in2
Vn_kN = (Vn_lb/1000)*4.448
phi_joint = 0.85
phiVn_kN = phi_joint*Vn_kN
print(f"\nVn = gamma*sqrt(fc')*Aj = {Vn_lb:.0f}lb = {Vn_kN:.2f}kN")
print(f"phiVn(phi={phi_joint}) = {phiVn_kN:.2f}kN")

ratio_joint = phiVn_kN/Vj_kN
print(f"\n檢核: Vj({Vj_kN:.2f}kN) <= phiVn({phiVn_kN:.2f}kN)? {Vj_kN <= phiVn_kN}")
print(f"餘裕倍數 = {ratio_joint:.2f}")

assert Vj_kN <= phiVn_kN, "接頭剪力應該通過"
print(f"\n[PASS] 梁柱接頭剪力檢核通過, 餘裕{ratio_joint:.2f}倍")
print("這裡用gamma=10(最不利的\"外側接頭, 無側向梁圍束\"分類)——")
print("這個2D單跨構架的每個接頭, 只有一根梁從單一方向匯入, 沒有")
print("垂直方向的梁提供額外圍束, 是保守但符合這個構架真實拓撲的假設")


T1 = As_top*1.25*fy = 5.73*5250.0 = 30082.5kgf = 295.01kN
Vcol(子構架平衡估計) = Mpr/((h1+h2)/2) = 123.00/3.5 = 35.14kN

Vj(接頭水平剪力需求) = T1 - Vcol = 295.01 - 35.14 = 259.87kN

有效接頭寬度 bj = min(70.0, 40.0) = 40.0cm
接頭面積 Aj = 1600.0cm^2
fc' = 280kgf/cm^2 = 3982.4psi, Aj = 248.00in^2

Vn = gamma*sqrt(fc')*Aj = 156505lb = 696.13kN
phiVn(phi=0.85) = 591.71kN

檢核: Vj(259.87kN) <= phiVn(591.71kN)? True
餘裕倍數 = 2.28

[PASS] 梁柱接頭剪力檢核通過, 餘裕2.28倍
這裡用gamma=10(最不利的"外側接頭, 無側向梁圍束"分類)——
這個2D單跨構架的每個接頭, 只有一根梁從單一方向匯入, 沒有
垂直方向的梁提供額外圍束, 是保守但符合這個構架真實拓撲的假設


## 小結:`Design Freeze` 前的完整規範閉環

- 柱:用 `Stage 2/3` 四工具驗證過的真實需求反推,發現 $\rho=0.02$
  這個從 `Case-06` 系列一開始就存在的假設值,遠遠超過實際需要
  (`utilization` 只有 `0.08~0.21`),真正 governing 的是 `ACI 318`
  最小配筋比規範(`1%`),不是強度需求
- 梁:同樣用 `Stage 2/3` 的規範等效靜力法需求設計,配筋遠比 `VL-14`
  當時反推的輕量——這不是矛盾,是設計層級(規範地震力)跟驗證層級
  (`pushover` 極限需求)本來就不該混用同一組配筋反推邏輯
- 強柱弱梁:用這次一致需求層級設計出的柱梁配筋,大幅通過(餘裕
  `3.22` 倍),跟 `Case-06.6` 那次「柱先降伏」的結果方向相反——
  印證了 `demand` 層級一致性的重要性
- 這是 `Design Freeze` 前的關鍵一步:確認柱梁配筋、強柱弱梁都通過
  之後,才能真正凍結這組設計結果,餵給後續的非線性模型